# ATOM PnP - ESP32 to ROS Communication Setup Guide

## System Architecture Overview

```bash

┌─────────────────────────┐         WiFi          ┌─────────────────────────┐
│      ESP32 Board        │ ←──────────────────→  │    Ubuntu VM (ROS)      │
│                         │                       │                         │
│  Publishes:             │                       │  Subscribes to:         │
│  - /motor_encoders      │ ───────────────────→  │  - /motor_encoders      │
│  - /robot_pose          │ ───────────────────→  │  - /robot_pose          │
│  - /esp32_debug         │ ───────────────────→  │  - /esp32_debug         │
│  - /esp32_heartbeat     │ ───────────────────→  │  - /esp32_heartbeat     │
│                         │                       │                         │
│  Subscribes to:         │                       │  Publishes:             │
│  - /motor_commands      │ ←───────────────────  │  - /motor_commands      │
│  - /esp32_command       │ ←───────────────────  │  - /esp32_command       │
│  - /cmd_vel             │ ←───────────────────  │  - /cmd_vel             │
└─────────────────────────┘                       └─────────────────────────┘
```


## Table of Contents
1. Network Setup & IP Configuration
2. Project Structure
3. ESP32 Code Files
4. ROS Package Structure
5. Communication Topics
6. Step-by-Step Testing
7. Troubleshooting
8. Quick Reference

## 1. Network Setup & IP Configuration

### Find Network Information

**On Ubuntu VM:**
```bash
# Get VM's IP address
ifconfig

On ESP32:

- IP displayed in Serial Monitor after WiFi connection

- Set statically in JOESWiFiSetup.h

## 1.2 IP Configuration Table


| Parameter    | Default Value     | Description                     |
|--------------|------------------|---------------------------------|
| wifi_ssid    | "WE_ACFAA2"      | WiFi network name              |
| wifi_pass    | "6988d5ab"       | WiFi password                  |
| ros_server   | 192.168.xxx.xx   | Ubuntu VM IP (ROS Master)      |
| esp_port     | 11411            | Communication port             |
| local_IP     | 192.168.xxx.xx   | ESP32 static IP                |
| gateway      | 192.168.1.1      | Network gateway                |
| subnet       | 255.255.255.0    | Subnet mask                    |

⚠️ Important: Update ros_server to match your Ubuntu VM's IP!

## 2. Project Structure

### 2.1 ESP32 Files

```bash
esp32_project/
├── main.ino                    # Main program loop
├── MotorBridge.h               # Motor & encoder function declarations
├── MotorBridge.cpp             # Motor & encoder implementations
├── WiFiHardware.h              # WiFi hardware abstraction for ROS
├── JOESWiFiSetup.h             # WiFi credentials and connection functions
└── platformio.ini              # PlatformIO configuration (if using)

### 2.2 ROS Package Structure

```bash
~/yusuf_ws/src/atom_pnp/
├── CMakeLists.txt              # Build configuration
├── package.xml                 # Package metadata
├── scripts/
│   ├── communication_tester.py # Test script for monitoring
│   ├── motor_commander.py      # Interactive motor control
│   └── ros_pid_controller.py   # PID controller (for later use)
└── launch/
    └── atom_pnp_test.launch    # Launch file for testing

## 3. ESP32 Code Files
### 3.1 WiFiHardware.h - WiFi Communication Layer
Purpose: Provides TCP communication interface between ESP32 and ROS using WiFi.
Key Functions:
- init() - Initialize TCP connection
- setConnection(server, port) - Set ROS server IP and port
- read() - Read data from ROS
- write(data, length) - Write data to ROS
- time() - Get current time in milliseconds

Key Variables:
```bash
    WiFiClient client;    // TCP client for communication
    IPAddress server;     // ROS server IP address
    uint16_t port;        // Communication port
``` 
How it works:
1. Creates a TCP client connection to the ROS server
2. Handles automatic reconnection if connection drops
3. Provides millisecond timing for ROS synchronization

### 3.2 JOESWiFiSetup.h - Network Configuration & Connection
Purpose: Manages WiFi connection and ROS node setup.

Key Functions:
```bash
connectWiFi()
```
- Configures ESP32 with static IP
- Connects to specified WiFi network
- Restarts ESP32 if connection fails after 5 seconds
```bash
connectROS()
```
- Sets up ROS communication parameters
- Initializes ROS node handle
- Subscribes to all required topics
- Advertises all publishers
- Waits for successful ROS master connection

Key Configuration:
```bash
    const char* wifi_ssid = "WE_ACFAA2";      // WiFi network name
    const char* wifi_pass = "6988d5ab";       // WiFi password
    IPAddress ros_server(192, 168, xxx, xx);  // ROS Master IP
    IPAddress local_IP(192, 168, xxx, xx);    // ESP32 static IP
    uint16_t esp_port = 11411;                 // Communication port
```



### 3.3 MotorBridge.h & MotorBridge.cpp - Hardware Interface
Purpose: Handles all motor control and encoder reading hardware interactions.

MotorBridge.h - Pin Definitions:
```bash
// Motor control pins
#define ENA 4       // Right motor PWM
#define IN1 16      // Right motor direction 1
#define IN2 17      // Right motor direction 2
#define ENB 5       // Left motor PWM
#define IN3 18      // Left motor direction 1
#define IN4 19      // Left motor direction 2

// Encoder pins
#define RIGHT_ENCODER_INTERRUPT 26  // Right encoder signal
#define RIGHT_ENCODER_DIRECTION 27  // Right encoder direction
#define LEFT_ENCODER_INTERRUPT 32   // Left encoder signal
#define LEFT_ENCODER_DIRECTION 33   // Left encoder direction

// Robot parameters
#define WHEEL_RADIUS 0.034    // Wheel radius in meters
#define WHEEL_DISTANCE 0.18   // Distance between wheels in meters
#define PPR 517               // Encoder Pulses Per Revolution
```
Key Functions:
| Function                          | Purpose                                      |
|----------------------------------|----------------------------------------------|
| setupMotorsBridge()              | Initialize motor pins, encoder pins, and interrupts |
| stopMotorsBridge()               | Stop all motors and reset state              |
| controlMotorsDirect(left_rpm, right_rpm) | Set motor speeds directly           |
| getEncoderReadings(data[])       | Get RPMs and encoder counts                  |
| updatePoseBridge()               | Calculate robot position from encoders       |
| getXPosition(), getYPosition(), getTheta() | Get current pose                 |



Encoder ISR (Interrupt Service Routine):
```bash
void IRAM_ATTR rightEncoderISR() {
    int direction = digitalRead(RIGHT_ENCODER_DIRECTION);
    right_encoder_ticks += (direction ? -1 : 1);
}
```
- Triggered on every encoder pulse
- Updates tick count based on direction
- Uses IRAM_ATTR for fast execution

Motor Control Flow:

```bash
    ROS Command (RPM) → controlMotorsDirect() → PWM + Direction pins → Motors
```
Encoder Reading Flow:
```bash
    Encoder Pulse → ISR → tick count → RPM calculation → Published to ROS
``` 

### 3.4 main.ino - Main Program
Purpose: Ties everything together, manages ROS communication loop.

Configurable Rates:
```bash
#define ENCODER_PUBLISH_RATE_MS    50    // How often to send data (50ms = 20Hz)
#define DEBUG_PUBLISH_RATE_MS      2000  // Debug message frequency (2 seconds)
#define HEARTBEAT_PUBLISH_RATE_MS  2000  // Heartbeat frequency (2 seconds)
#define ROS_SPIN_RATE_MS           10    // ROS message processing (10ms = 100Hz)
```
Publishers (ESP32 → ROS):
| Publisher       | Topic              | Message Type           | Data Sent                                      |
|-----------------|--------------------|------------------------|-----------------------------------------------|
| pose_pub        | /robot_pose        | Pose2D                | x, y, theta                                   |
| encoder_pub     | /motor_encoders    | Float32MultiArray     | [left_rpm, right_rpm, left_count, right_count] |
| debug_pub       | /esp32_debug       | String                | Status message with RPMs and position         |
| heartbeat_pub   | /esp32_heartbeat   | Int32                 | Incrementing counter                          |

Subscribers (ROS → ESP32):
| Subscriber  | Topic            | Message Type        | Action                     |
|-------------|------------------|---------------------|----------------------------|
| motor_sub   | /motor_commands  | Float32MultiArray   | Sets motor RPMs            |
| debug_sub   | /esp32_command   | String              | Prints message to Serial   |
| vel_sub     | /cmd_vel         | Twist               | Legacy compatibility       |

Main Loop Flow:
1. Check WiFi connection
2. Process incoming ROS messages (nh.spinOnce())
3. If encoder interval reached → publish encoder data
4. If debug interval reached → publish debug data
5. Delay and repeat

## 4. ROS Package Structure
### 4.1 ROS Workspace: yusuf_ws
  - Location: /home/vboxuser/yusuf_ws/
  - Package Name: atom_pnp

### 4.2 CMakeLists.txt - Build Configuration
```bash
cmake_minimum_required(VERSION 3.0.2)
project(atom_pnp)

# Dependencies
find_package(catkin REQUIRED COMPONENTS
  geometry_msgs    # For Pose2D, Twist messages
  roscpp           # C++ ROS library
  rospy            # Python ROS library
  std_msgs         # For String, Int32, Float32MultiArray
)

# Install Python scripts
catkin_install_python(PROGRAMS
  scripts/communication_tester.py
  scripts/motor_commander.py
  scripts/ros_pid_controller.py
  DESTINATION ${CATKIN_PACKAGE_BIN_DESTINATION}
)

# Install launch files
install(DIRECTORY launch/
  DESTINATION ${CATKIN_PACKAGE_SHARE_DESTINATION}/launch
)
```
### 4.3 package.xml - Package Metadata
```bash 
<package format="2">
  <name>atom_pnp</name>
  <version>0.0.1</version>
  <description>ATOM Pick and Place - ESP32 to ROS Communication Bridge</description>
  <maintainer email="yusuf@example.com">Yusuf</maintainer>
  <license>MIT</license>
  
  <buildtool_depend>catkin</buildtool_depend>
  <depend>geometry_msgs</depend>
  <depend>roscpp</depend>
  <depend>rospy</depend>
  <depend>std_msgs</depend>
</package>
```

## 5. ROS Python Scripts
### 5.1 communication_tester.py
- Purpose: Monitors all communication between ROS and ESP32.
- Features:
    1. Subscribes to all ESP32 published topics
    2. Can send automated test messages
    3. Displays encoder data and robot pose

- Usage:
    ```bash
    # Automated test mode
    rosrun atom_pnp communication_tester.py auto

    # Interactive mode (just listens)
    rosrun atom_pnp communication_tester.py
    ```

### 5.2 motor_commander.py
- Purpose: Interactive motor control interface.
- Commands:

| Command   | Action                          |
|-----------|---------------------------------|
| FORWARD   | Both motors at 50 RPM           |
| BACKWARD  | Both motors at -50 RPM          |
| LEFT      | Turn left (-30, 30 RPM)         |
| RIGHT     | Turn right (30, -30 RPM)        |
| STOP      | Stop motors                     |
| 50 30     | Custom left and right RPM       |
| q         | Quit and stop motors            |

- Usage:
```bash 
rosrun atom_pnp motor_commander.py
``` 
### 5.3 ros_pid_controller.py
- Purpose: PID controller for autonomous navigation (for later use).



## 6. Communication Topics Reference
### 6.1 Complete Topic List
| Topic              | Direction        | Message Type                     | Frequency                | Purpose                                      |
|--------------------|------------------|----------------------------------|--------------------------|----------------------------------------------|
| /esp32_debug       | ESP32 → ROS      | std_msgs/String                 | Every 2s (configurable)  | Status messages with sensor data             |
| /esp32_heartbeat   | ESP32 → ROS      | std_msgs/Int32                  | Every 2s (configurable)  | Connection health check                      |
| /motor_encoders    | ESP32 → ROS      | std_msgs/Float32MultiArray      | 20Hz (configurable)      | Motor RPMs and encoder counts                |
| /robot_pose        | ESP32 → ROS      | geometry_msgs/Pose2D            | 20Hz (configurable)      | Robot position (x, y, theta)                 |
| /esp32_command     | ROS → ESP32      | std_msgs/String                 | On demand                | Debug messages to ESP32                      |
| /motor_commands    | ROS → ESP32      | std_msgs/Float32MultiArray      | On demand                | Target RPMs for motors                       |
| /cmd_vel           | ROS → ESP32      | geometry_msgs/Twist             | On demand                | Legacy velocity commands                     |


### 6.2 Message Formats
/motor_encoders (Float32MultiArray):
```bash
    data[0] = Left motor RPM
    data[1] = Right motor RPM
    data[2] = Left encoder count
    data[3] = Right encoder count
```
/motor_commands (Float32MultiArray):
```bash
    data[0] = Target left motor RPM
    data[1] = Target right motor RPM
```
/robot_pose (Pose2D):
```bash
    x     = X position in meters
    y     = Y position in meters
    theta = Orientation in radians
```

## 7. Step-by-Step Testing Procedure
### 7.1 Prerequisites
- ESP32 connected to computer via USB
- Ubuntu VM running
- Both on same WiFi network
- Arduino IDE or PlatformIO configured for ESP32

### 7.2 Find IP Addresses
```bash
# On Ubuntu VM - Terminal 1
ifconfig
# Note the inet address (e.g., 192.168.xxx.xx)
```

### 7.3 Configure ESP32
1. Open JOESWiFiSetup.h
2. Update ros_server with Ubuntu VM IP:
```bash 
        IPAddress ros_server(192, 168, xxx, xx);  // Msh hadeek el IP bta3y la2 😜
```
3. Verify WiFi credentials
4. Upload code to ESP32
5. Open Serial Monitor (115200 baud)

### 7.4 Build ROS Package
```bash
    # Ubuntu VM - Terminal 1
    cd ~/yusuf_ws
    catkin_make
    source devel/setup.bash
```
### 7.5 Start ROS Master
```bash
    # Ubuntu VM - Terminal 1
    roscore
```
### 7.6 Test ESP32 → ROS Communication
Step 1: Monitor debug messages

```bash
    # Ubuntu VM - Terminal 2
    source ~/yusuf_ws/devel/setup.bash
    rostopic echo /esp32_debug
```
*Expected: See "ESP32 Alive!" messages every 2 seconds*

Step 2: Monitor heartbeat

```bash
    # Ubuntu VM - Terminal 3
    source ~/yusuf_ws/devel/setup.bash
    rostopic echo /esp32_heartbeat
``` 
*Expected: See incrementing numbers (0, 1, 2, 3...)*

Step 3: Monitor encoder data

``` bash
    # Ubuntu VM - Terminal 4
    source ~/yusuf_ws/devel/setup.bash
    rostopic echo /motor_encoders
```
*Expected: See RPM and count data arrays*

Step 4: Monitor robot pose

```bash
    # Ubuntu VM - Terminal 5
    source ~/yusuf_ws/devel/setup.bash
    rostopic echo /robot_pose
```
*Expected: See x, y, theta values*

### 7.7 Test ROS → ESP32 Communication
Step 5: Send test message to ESP32

```bash
    # Ubuntu VM - Terminal 6
    source ~/yusuf_ws/devel/setup.bash
    rostopic pub /esp32_command std_msgs/String "data: 'Hello ESP32! Test message from ROS!'"
```
*Expected on ESP32 Serial:*
```bash
╔════════════════════════════════════════╗
║     📨 MESSAGE RECEIVED FROM ROS      ║
╠════════════════════════════════════════╣
║ Hello ESP32! Test message from ROS!   ║
╚════════════════════════════════════════╝
``` 
Step 6: Send motor commands

```bash
    # Ubuntu VM - Terminal 6
    rostopic pub /motor_commands std_msgs/Float32MultiArray "data: [50.0, 50.0]"
```
*Expected on ESP32 Serial:*
```bash
╔════════════════════════════════════════╗
║      ⚙️  MOTOR COMMAND RECEIVED       ║
╠════════════════════════════════════════╣
║ Left RPM:  50.00                      ║
║ Right RPM: 50.00                      ║
╚════════════════════════════════════════╝
```
Step 7: Stop motors

``` bash
    rostopic pub /motor_commands std_msgs/Float32MultiArray "data: [0.0, 0.0]"
```
### 7.8 Use Automated Test Script
```bash
    # Ubuntu VM
    rosrun atom_pnp communication_tester.py auto
```
### 7.9 Use Interactive Motor Control
```bash
    # Ubuntu VM
    rosrun atom_pnp motor_commander.py
```

## 8. Troubleshooting
### 8.1 ESP32 Can't Connect to WiFi
Symptom: Serial shows "Connecting to Wi-Fi..." repeatedly

Solutions:
- Check WiFi SSID and password in JOESWiFiSetup.h
- Verify WiFi router is working
- Check ESP32 is in range

### 8.2 ESP32 Can't Connect to ROS Master
Symptom: "Connected to Wi-Fi" but no ROS connection

Solutions:
- Verify ros_server IP matches Ubuntu VM
- Check both devices on same network
- Ensure roscore is running on Ubuntu VM
- Try pinging ESP32 from Ubuntu: ping 192.168.xxx.xx

### 8.3 No Topics Showing in ROS
```bash
    # Check all topics
    rostopic list

    # If no topics appear, check ROS master connection
    rostopic list | grep esp32
```
### 8.4 Verify Communication
```bash
    # Check if ROS master is accessible
    rosnode list

    # Check topic details
    rostopic info /esp32_debug

    # See topic bandwidth
    rostopic bw /motor_encoders

    # See topic publishing rate
    rostopic hz /esp32_heartbeat
```

## Summary
## Communication Flow Diagram

```bash
                   ESP32 PUBLISHES                       ROS RECEIVES
                    ─────────────►                      ─────────────►
                    
/esp32_debug ─────── Debug string ──────► Terminal shows ESP32 status
/esp32_heartbeat ─── Counter ───────────► Terminal shows connection alive
/motor_encoders ──── RPMs + Counts ─────► Terminal shows motor data
/robot_pose ──────── X, Y, Theta ───────► Terminal shows position


                    ROS PUBLISHES                        ESP32 RECEIVES
                    ─────────────►                      ─────────────►

/esp32_command ───── Text message ──────► Serial prints message
/motor_commands ──── Left/Right RPM ────► Motors spin at target speed
/cmd_vel ─────────── Linear/Angular ────► (Legacy compatibility)
```
## File Responsibilities Matrix

| File                     | Responsibility                                 |
|--------------------------|-----------------------------------------------|
| WiFiHardware.h           | TCP/IP communication layer                    |
| JOESWiFiSetup.h          | WiFi connection, ROS initialization           |
| MotorBridge.h/.cpp       | Hardware abstraction (motors, encoders)       |
| main.ino                 | Main loop, rate control, topic management     |
| communication_tester.py  | Monitor all ESP32 communication               |
| motor_commander.py       | Manual motor control interface                |
| ros_pid_controller.py    | Autonomous control (future use)               |

## Key Configuration Points
1. ESP32 IP: 192.168.xxx.xx (in JOESWiFiSetup.h)
2. ROS Master IP: 192.168.xxx.xx (in JOESWiFiSetup.h)
3. Publish rates: Configurable via #define in main.ino
4. Motor pins: Defined in MotorBridge.h
5. Robot dimensions: Wheel radius, wheel distance in MotorBridge.h

## Quick Reference Commands

```bash
    # Build ROS package
    cd ~/yusuf_ws && catkin_make && source devel/setup.bash

    # Start ROS
    roscore

    # Monitor ESP32
    rostopic echo /esp32_debug

    # Send message to ESP32
    rostopic pub /esp32_command std_msgs/String "data: 'Your message'"

    # Control motors
    rostopic pub /motor_commands std_msgs/Float32MultiArray "data: [50.0, 50.0]"

    # Stop motors
    rostopic pub /motor_commands std_msgs/Float32MultiArray "data: [0.0, 0.0]"

    # List all topics
    rostopic list | grep -E "esp32|motor|robot"

    # Run test script
    rosrun atom_pnp communication_tester.py auto

    # Interactive motor control
    rosrun atom_pnp motor_commander.py

```

**Document Version:** 1.0

**Last Updated:** April 2026

**Authors:** Yusuf M. Abulfotouh

**Project:** ATOM PnP - Pick and Place Robot